In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Connect to the DuckDB warehouse. Works whether the kernel's working dir is
# notebooks/ or the project root (VSCode Jupyter default).
for _p in [Path('../data/retail.duckdb'), Path('data/retail.duckdb')]:
    if _p.exists():
        DB_PATH = _p
        break
else:
    raise FileNotFoundError('retail.duckdb not found under ../data/ or data/')

con = duckdb.connect(str(DB_PATH), read_only=True)
print(f'Connected to {DB_PATH.resolve()}')

# SQL Exploration - Star Schema & Analytics

Run SQL queries directly against your DuckDB data warehouse. No dbt, no command line.

Press `Shift+Enter` on each cell to see results.

### 1. What Tables Do We Have?

In [2]:
tables = con.sql('SHOW TABLES').df()
results = []
for _, row in tables.iterrows():
    name = row['name']
    try:
        count = con.sql(f'SELECT COUNT(*) FROM {name}').fetchone()[0]
        cols = con.sql(f'DESCRIBE {name}').df()
        col_list = ', '.join(cols['column_name'].tolist())
        results.append({'Table': name, 'Rows': f'{count:,}', 'Columns': col_list})
    except Exception:
        results.append({'Table': name, 'Rows': 'view', 'Columns': '(staging view)'})
display(pd.DataFrame(results))

,Table,Rows,Columns
0,dim_country,43,"country_sk, country_name"
1,dim_customer,"5,922","customer_sk, customer_id, first_seen_date, seg..."
2,dim_date,599,"date_sk, date, year, month, week, day_of_week,..."
3,dim_product,"4,759","product_sk, stock_code, description"
4,fact_transactions,"1,008,442","transaction_sk, customer_sk, product_sk, count..."
5,int_transactions__prepared,"1,008,442","invoice, stock_code, description, quantity, pr..."
6,mart_rfm,"5,860","customer_id, recency_days, frequency, monetary..."
7,stg_silver__transactions,"1,008,442","invoice, stock_code, description, quantity, pr..."


### 📐 Star Schema Overview

```
                    ┌─────────────────────┐
                    │     dim_date         │
                    │  PK  date_sk         │
                    │  • date, year, month │
                    │  • week, quarter     │
                    └────────┬────────────┘
                             │ date_sk
                             │
┌─────────────────────┐      │      ┌─────────────────────┐
│   dim_customer       │      │      │   dim_product        │
│  PK  customer_sk     │      │      │  PK  product_sk      │
│  • customer_id       │      │      │  • stock_code        │
│  • segment           │      │      │  • description       │
└────────┬────────────┘      │      └────────┬────────────┘
         │ customer_sk       │               │ product_sk
         │                   │               │
         └───────┬───────────┴───────┬───────┘
                 ▼                   ▼
┌──────────────────────────────────────────┐     ┌─────────────────────┐
│        fact_transactions                 │     │    dim_country       │
│  PK   transaction_sk                     │◄────│  PK  country_sk      │
│  FK   customer_sk, product_sk            │     │  • country_name      │
│  FK   country_sk, date_sk                │     └─────────────────────┘
│  MEASURES  quantity, price, line_amount  │──┐   country_sk
└──────────────────────────────────────────┘  │
                                               ▼
                                 ┌─────────────────────┐
                                 │     mart_rfm         │
                                 │  • customer_id       │
                                 │  • recency_days      │
                                 │  • frequency         │
                                 │  • monetary, segment │
                                 └─────────────────────┘
```

### 2. Peek at Each Table

In [3]:
# dim_date
con.sql('SELECT * FROM dim_date ORDER BY date_sk LIMIT 5').df()

,date_sk,date,year,month,week,day_of_week,quarter
0,1,2024-06-02,2024,6,22,0,2
1,2,2024-06-03,2024,6,23,1,2
2,3,2024-06-04,2024,6,23,2,2
3,4,2024-06-05,2024,6,23,3,2
4,5,2024-06-06,2024,6,23,4,2


In [4]:
# dim_customer
con.sql('SELECT * FROM dim_customer ORDER BY customer_sk LIMIT 5').df()

,customer_sk,customer_id,first_seen_date,segment
0,0,,NaT,Unknown
1,1,12346,2024-06-15 08:34:00,UNKNOWN
2,2,12347,2025-05-02 14:20:00,UNKNOWN
3,3,12348,2025-03-29 14:59:00,UNKNOWN
4,4,12349,2024-06-05 12:49:00,UNKNOWN


In [5]:
# dim_product
con.sql('SELECT * FROM dim_product ORDER BY product_sk LIMIT 5').df()

,product_sk,stock_code,description
0,1,10002,INFLATABLE POLITICAL GLOBE
1,2,10002R,ROBOT PENCIL SHARPNER
2,3,10080,GROOVY CACTUS INFLATABLE
3,4,10109,BENDY COLOUR PENCILS
4,5,10120,DOGGY RUBBER


In [6]:
# dim_country
con.sql('SELECT * FROM dim_country ORDER BY country_sk LIMIT 10').df()

,country_sk,country_name
0,1,Australia
1,2,Austria
2,3,Bahrain
3,4,Belgium
4,5,Bermuda
5,6,Brazil
6,7,Canada
7,8,Channel Islands
8,9,Cyprus
9,10,Czech Republic


In [7]:
# fact_transactions
con.sql('SELECT * FROM fact_transactions ORDER BY transaction_sk LIMIT 5').df()

,transaction_sk,customer_sk,product_sk,country_sk,date_sk,quantity,price,line_amount,is_cancellation
0,1,735,4093,41,1,12.00,6.95,83.40,False
1,2,735,3407,41,1,12.00,6.75,81.00,False
2,3,735,3409,41,1,12.00,6.75,81.00,False
3,4,735,1284,41,1,48.00,2.10,100.80,False
4,5,735,629,41,1,24.00,1.25,30.00,False


In [8]:
# mart_rfm - Top 10 customers by monetary value
con.sql('SELECT * FROM mart_rfm ORDER BY monetary DESC LIMIT 10').df()

,customer_id,recency_days,frequency,monetary,r_score,f_score,m_score,segment
0,18102,6,142,"569,501.50",5,5,5,Champions
1,14646,11,148,"516,874.50",5,5,5,Champions
2,14156,4,156,"313,437.62",5,5,5,Champions
3,14911,3,391,"285,118.84",5,5,5,Champions
4,17450,3,51,"244,784.25",5,5,5,Champions
5,13694,20,140,"192,509.53",4,5,5,Champions
6,17511,11,58,"164,753.55",5,5,5,Champions
7,12415,19,28,"144,458.37",4,5,5,Champions
8,16684,6,54,"141,740.79",5,5,5,Champions
9,15061,5,126,"122,493.16",5,5,5,Champions


### 3. Full Star Schema Join - Fact + All Dimensions

In [9]:
con.sql("""
SELECT 
    f.transaction_sk,
    c.customer_id,
    c.segment         AS customer_segment,
    p.stock_code,
    p.description,
    cy.country_name,
    d.date             AS invoice_date,
    d.year,
    d.quarter,
    f.quantity,
    f.price,
    f.line_amount
FROM fact_transactions f
LEFT JOIN dim_customer c  ON f.customer_sk = c.customer_sk
LEFT JOIN dim_product  p  ON f.product_sk  = p.product_sk
LEFT JOIN dim_country  cy ON f.country_sk  = cy.country_sk
LEFT JOIN dim_date     d  ON f.date_sk     = d.date_sk
ORDER BY f.line_amount DESC
LIMIT 15
""").df()

,transaction_sk,customer_id,customer_segment,stock_code,description,country_name,invoice_date,year,quarter,quantity,price,line_amount
0,553505,12346,UNKNOWN,23166,MEDIUM CERAMIC TOP STORAGE JAR,United Kingdom,2025-07-20,2025,3,"74,215.00",1.04,"77,183.60"
1,712264,15098,UNKNOWN,22502,PICNIC BASKET WICKER 60 PIECES,United Kingdom,2025-12-10,2025,4,60.00,649.50,"38,970.00"
2,236735,,Unknown,M,Manual,United Kingdom,2024-12-17,2024,4,1.00,"25,111.09","25,111.09"
3,423634,15838,UNKNOWN,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,United Kingdom,2025-05-06,2025,2,"9,360.00",1.69,"15,818.40"
4,507461,,Unknown,AMAZONFEE,AMAZON FEE,United Kingdom,2025-06-08,2025,2,1.00,"13,541.33","13,541.33"
5,788747,,Unknown,B,Adjust bad debt,United Kingdom,2026-02-11,2026,1,1.00,"11,062.06","11,062.06"
6,132217,,Unknown,M,Manual,United Kingdom,2024-09-22,2024,3,1.00,"10,953.50","10,953.50"
7,132215,12918,UNKNOWN,M,Manual,United Kingdom,2024-09-22,2024,3,1.00,"10,953.50","10,953.50"
8,335264,,Unknown,M,Manual,United Kingdom,2025-03-18,2025,1,1.00,"10,468.80","10,468.80"
9,351505,14063,UNKNOWN,M,Manual,United Kingdom,2025-03-29,2025,1,1.00,"10,468.80","10,468.80"


### 4. RFM Segment Summary

In [10]:
con.sql("""
SELECT 
    segment,
    COUNT(*)                    AS customers,
    ROUND(AVG(recency_days), 1)  AS avg_recency_days,
    ROUND(AVG(frequency), 1)     AS avg_frequency,
    ROUND(AVG(monetary), 2)      AS avg_monetary,
    ROUND(SUM(monetary), 2)      AS total_monetary,
    ROUND(SUM(monetary) * 100.0 / SUM(SUM(monetary)) OVER (), 1) AS pct_revenue
FROM mart_rfm
GROUP BY segment
ORDER BY total_monetary DESC
""").df()

,segment,customers,avg_recency_days,avg_frequency,avg_monetary,total_monetary,pct_revenue
0,Champions,1238,15.80,17.20,"9,110.16","11,278,382.20",66.50
1,Loyal Customers,736,51.00,6.60,"2,571.03","1,892,276.56",11.20
2,Potential Loyalists,571,110.40,4.80,"1,788.85","1,021,435.28",6.00
3,At Risk,810,317.30,3.10,"1,031.02","835,126.10",4.90
4,Cannot Lose Them,150,415.10,10.30,"4,926.98","739,047.28",4.40
5,Hibernating,553,482.20,1.90,707.94,"391,489.43",2.30
6,Need Attention,551,52.60,2.00,685.85,"377,902.86",2.20
7,About to Sleep,545,212.40,1.00,356.67,"194,385.31",1.10
8,Lost,520,611.70,1.00,317.84,"165,278.61",1.00
9,Promising,186,31.30,1.00,331.06,"61,577.32",0.40


### 5. Monthly Revenue Trend

In [11]:
con.sql("""
SELECT 
    d.year,
    d.month,
    COUNT(DISTINCT f.transaction_sk)  AS transactions,
    COUNT(DISTINCT f.customer_sk)     AS unique_customers,
    ROUND(SUM(f.line_amount), 2)      AS revenue,
    ROUND(SUM(f.line_amount) / COUNT(DISTINCT f.transaction_sk), 2) AS aov
FROM fact_transactions f
JOIN dim_date d ON f.date_sk = d.date_sk
WHERE f.line_amount > 0
GROUP BY d.year, d.month
ORDER BY d.year, d.month
""").df()

,year,month,transactions,unique_customers,revenue,aov
0,2024,6,43453,956,"822,483.95",18.93
1,2024,7,28951,682,"632,554.76",21.85
2,2024,8,30490,836,"596,261.06",19.56
3,2024,9,38670,1026,"805,132.85",20.82
4,2024,10,32858,943,"678,875.25",20.66
5,2024,11,33383,967,"657,705.50",19.70
6,2024,12,40185,1088,"779,716.89",19.40
7,2025,1,31180,891,"649,268.64",20.82
8,2025,2,29505,869,"628,701.22",21.31
9,2025,3,40291,1119,"906,130.97",22.49


### 6. Top 10 Products by Revenue

In [12]:
con.sql("""
SELECT 
    p.stock_code,
    p.description,
    COUNT(*)                        AS times_sold,
    ROUND(SUM(f.quantity), 0)        AS total_quantity,
    ROUND(SUM(f.line_amount), 2)     AS total_revenue,
    ROUND(SUM(f.line_amount) * 100.0 / SUM(SUM(f.line_amount)) OVER (), 2) AS pct_revenue
FROM fact_transactions f
JOIN dim_product p ON f.product_sk = p.product_sk
WHERE f.line_amount > 0
GROUP BY p.stock_code, p.description
ORDER BY total_revenue DESC
LIMIT 10
""").df()

,stock_code,description,times_sold,total_quantity,total_revenue,pct_revenue
0,M,Manual,846,"8,825.00","339,187.40",1.70
1,22423,REGENCY CAKESTAND 3 TIER,3906,"26,083.00","325,341.37",1.63
2,DOT,DOTCOM POSTAGE,1402,"1,402.00","292,186.57",1.46
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,5559,"94,043.00","259,231.33",1.30
4,85099B,JUMBO BAG RED RETROSPOT,4027,"96,713.00","181,654.69",0.91
5,47566,PARTY BUNTING,2686,"28,118.00","147,731.34",0.74
6,84879,ASSORTED COLOUR BIRD ORNAMENT,2818,"79,384.00","128,153.03",0.64
7,POST,POSTAGE,1812,"5,267.00","123,556.13",0.62
8,22086,PAPER CHAIN KIT 50'S CHRISTMAS,1988,"33,576.00","112,540.45",0.56
9,23166,MEDIUM CERAMIC TOP STORAGE JAR,240,"77,856.00","81,492.97",0.41


### 7. Revenue by Country

In [13]:
con.sql("""
SELECT 
    cy.country_name,
    COUNT(*)                        AS transactions,
    ROUND(SUM(f.line_amount), 2)     AS revenue,
    ROUND(SUM(f.line_amount) * 100.0 / SUM(SUM(f.line_amount)) OVER (), 1) AS pct
FROM fact_transactions f
JOIN dim_country cy ON f.country_sk = cy.country_sk
WHERE f.line_amount > 0
GROUP BY cy.country_name
ORDER BY revenue DESC
LIMIT 10
""").df()

,country_name,transactions,revenue,pct
0,United Kingdom,908931,"16,931,306.31",84.80
1,EIRE,16869,"652,465.34",3.30
2,Netherlands,4999,"542,310.07",2.70
3,Germany,16153,"417,798.41",2.10
4,France,13412,"345,671.81",1.70
5,Australia,1789,"169,283.46",0.80
6,Spain,3597,"108,016.28",0.50
7,Switzerland,3122,"100,685.59",0.50
8,Sweden,1335,"91,631.82",0.50
9,Denmark,767,"68,411.79",0.30


### 8. Customer Churn Analysis (90-day threshold)

In [14]:
con.sql("""
SELECT 
    segment,
    COUNT(*)                                AS total,
    SUM(CASE WHEN recency_days > 90 THEN 1 ELSE 0 END)  AS churned,
    ROUND(
        SUM(CASE WHEN recency_days > 90 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 
        1
    ) AS churn_rate_pct,
    ROUND(AVG(monetary), 2)                 AS avg_monetary
FROM mart_rfm
GROUP BY segment
ORDER BY churn_rate_pct DESC
""").df()

,segment,total,churned,churn_rate_pct,avg_monetary
0,At Risk,810,810.00,100.00,"1,031.02"
1,Cannot Lose Them,150,150.00,100.00,"4,926.98"
2,Lost,520,520.00,100.00,317.84
3,Hibernating,553,553.00,100.00,707.94
4,About to Sleep,545,435.00,79.80,356.67
5,Potential Loyalists,571,324.00,56.70,"1,788.85"
6,Need Attention,551,113.00,20.50,685.85
7,Loyal Customers,736,90.00,12.20,"2,571.03"
8,Champions,1238,0.00,0.00,"9,110.16"
9,Promising,186,0.00,0.00,331.06


### 9. Write Your Own Query

Edit the SQL below and run it!

In [15]:
my_query = """
SELECT 
    c.segment,
    d.year,
    d.quarter,
    ROUND(SUM(f.line_amount), 2) AS revenue
FROM fact_transactions f
JOIN dim_customer c ON f.customer_sk = c.customer_sk
JOIN dim_date     d ON f.date_sk     = d.date_sk
WHERE c.customer_sk != 0
  AND f.line_amount > 0
GROUP BY c.segment, d.year, d.quarter
ORDER BY c.segment, d.year, d.quarter
"""

con.sql(my_query).df()

,segment,year,quarter,revenue
0,UNKNOWN,2024,2,"683,504.01"
1,UNKNOWN,2024,3,"1,757,340.10"
2,UNKNOWN,2024,4,"1,855,016.53"
3,UNKNOWN,2025,1,"1,954,566.13"
4,UNKNOWN,2025,2,"2,807,573.33"
5,UNKNOWN,2025,3,"1,608,267.99"
6,UNKNOWN,2025,4,"1,817,921.97"
7,UNKNOWN,2026,1,"2,141,534.26"
8,UNKNOWN,2026,2,"2,331,176.62"


In [16]:
con.close()
print('Connection closed.')

Connection closed.
